In [0]:
from datetime import datetime, timezone

SOURCE_SYSTEM = "BCB_SGS"
DATASET_NAME = "bcb_sgs_selic"
STEP_NAME = "DQ_BRONZE"

execution_id = dbutils.jobs.taskValues.get(
    taskKey="INITIALIZE_MONITORING",
    key="execution_id"
)

step_start_timestamp = datetime.now(timezone.utc)

print("Execution ID:", execution_id)
print("Step:", STEP_NAME)

In [0]:
spark.sql(f"""
UPDATE workspace.brazilian_economic_monitoring.pipeline_step_execution
SET
    status = 'RUNNING',
    start_timestamp = CURRENT_TIMESTAMP()
WHERE execution_id = '{execution_id}'
  AND step_name = '{STEP_NAME}'
""")

print(f"{STEP_NAME} status: RUNNING")

In [0]:
try:

    # ---------------------------------------------------------
    # 1. Execute Bronze data quality checks
    # ---------------------------------------------------------

    spark.sql(f"""
        WITH quality_checks AS (

            -- 1. Null reference date
            SELECT
                'NULL_REFERENCE_DATE' AS check_name,
                COUNT(*) AS failed_records
            FROM workspace.brazilian_economic_bronze.bcb_sgs_selic
            WHERE reference_date_raw IS NULL

            UNION ALL

            -- 2. Null Selic value
            SELECT
                'NULL_VALUE' AS check_name,
                COUNT(*) AS failed_records
            FROM workspace.brazilian_economic_bronze.bcb_sgs_selic
            WHERE value_raw IS NULL

            UNION ALL

            -- 3. Invalid reference date
            SELECT
                'INVALID_REFERENCE_DATE' AS check_name,
                COUNT(*) AS failed_records
            FROM workspace.brazilian_economic_bronze.bcb_sgs_selic
            WHERE reference_date_raw IS NOT NULL
              AND TRY_TO_DATE(reference_date_raw, 'dd/MM/yyyy') IS NULL

            UNION ALL

            -- 4. Invalid numeric value
            SELECT
                'INVALID_VALUE' AS check_name,
                COUNT(*) AS failed_records
            FROM workspace.brazilian_economic_bronze.bcb_sgs_selic
            WHERE value_raw IS NOT NULL
              AND TRY_CAST(value_raw AS DECIMAL(8,4)) IS NULL

            UNION ALL

            -- 5. Duplicates inside the same ingestion execution
            SELECT
                'DUPLICATE_WITHIN_EXECUTION' AS check_name,
                COALESCE(SUM(record_count - 1), 0) AS failed_records
            FROM (
                SELECT
                    execution_id,
                    source_series_code,
                    reference_date_raw,
                    COUNT(*) AS record_count
                FROM workspace.brazilian_economic_bronze.bcb_sgs_selic
                GROUP BY
                    execution_id,
                    source_series_code,
                    reference_date_raw
                HAVING COUNT(*) > 1
            )
        )

        INSERT INTO workspace.brazilian_economic_governance.data_quality_results

        SELECT
            CURRENT_TIMESTAMP() AS check_timestamp,
            '{SOURCE_SYSTEM}' AS source_system,
            '{DATASET_NAME}' AS dataset_name,
            'BRONZE' AS layer,
            check_name,
            CASE
                WHEN failed_records = 0 THEN 'PASS'
                ELSE 'FAIL'
            END AS check_status,
            failed_records,
            '{execution_id}' AS execution_id
        FROM quality_checks
    """)

    print("Bronze data quality checks executed.")

    # ---------------------------------------------------------
    # 2. Count failed checks
    # ---------------------------------------------------------

    failed_checks = spark.sql(f"""
        SELECT COUNT(*) AS failed_checks
        FROM workspace.brazilian_economic_governance.data_quality_results
        WHERE execution_id = '{execution_id}'
          AND layer = 'BRONZE'
          AND check_status = 'FAIL'
    """).collect()[0]["failed_checks"]

    print("Failed checks:", failed_checks)

    # ---------------------------------------------------------
    # 3. Critical DQ failure
    # ---------------------------------------------------------

    if failed_checks > 0:
        raise ValueError(
            f"{failed_checks} critical Bronze data quality check(s) failed"
        )

    # ---------------------------------------------------------
    # 4. SUCCESS
    # ---------------------------------------------------------

    step_end_timestamp = datetime.now(timezone.utc)

    duration_seconds = int(
        (step_end_timestamp - step_start_timestamp).total_seconds()
    )

    spark.sql(f"""
        UPDATE workspace.brazilian_economic_monitoring.pipeline_step_execution
        SET
            status = 'SUCCESS',
            end_timestamp = CURRENT_TIMESTAMP(),
            duration_seconds = {duration_seconds},
            failed_checks = 0,
            error_message = NULL
        WHERE execution_id = '{execution_id}'
          AND step_name = '{STEP_NAME}'
    """)

    print(f"{STEP_NAME} status: SUCCESS")
    print("Duration seconds:", duration_seconds)

except Exception as e:

    # ---------------------------------------------------------
    # 5. FAILED
    # ---------------------------------------------------------

    step_end_timestamp = datetime.now(timezone.utc)

    duration_seconds = int(
        (step_end_timestamp - step_start_timestamp).total_seconds()
    )

    error_message = str(e).replace("'", "''")[:4000]

    spark.sql(f"""
        UPDATE workspace.brazilian_economic_monitoring.pipeline_step_execution
        SET
            status = 'FAILED',
            end_timestamp = CURRENT_TIMESTAMP(),
            duration_seconds = {duration_seconds},
            failed_checks = (
                SELECT COUNT(*)
                FROM workspace.brazilian_economic_governance.data_quality_results
                WHERE execution_id = '{execution_id}'
                  AND layer = 'BRONZE'
                  AND check_status = 'FAIL'
            ),
            error_message = '{error_message}'
        WHERE execution_id = '{execution_id}'
          AND step_name = '{STEP_NAME}'
    """)

    print(f"{STEP_NAME} status: FAILED")
    print("Error:", str(e))

    raise